In [1]:
import os
import sys
os.chdir('/zhome/71/c/146676/texture_tomography')
sys.path.append('/zhome/71/c/146676/texture_tomography/package/odf_mumott')
import numpy as np

from package.texture_tomography.operators.pfo_and_projection_batched_opencl import PFO_OPENCL_BATCHED
from package.texture_tomography.operators.operator_memory_model import OperatorMemoryModel
from package.texture_tomography.operators.memory_tracker import MemoryCounter
from package.texture_tomography.optimization.fista_opencl import FISTAOpenCL
from package.texture_tomography.optimization.fista_memory_model import FISTAMemoryModel
from package.texture_tomography.material import Material
from package.texture_tomography.multiresolution_refiner import OrientationTree, OrientationNode, generate_hopf_grid_fzone, invert_grid


import matplotlib.pyplot as plt
from pathlib import Path

import yaml

INFO:Setting the number of threads to 8. If your physical cores are fewer than this number, you may want to use numba.set_num_threads(n), and os.environ["OPENBLAS_NUM_THREADS"] = f"{n}" to set the number of threads to the number of physical cores n.
INFO:Setting numba log level to WARNING.


In [2]:
config_path = "configs/aluminum_config_small.yaml"
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)
    
cif_path = cfg["cif_path"]
N_theta = cfg['N_theta']
N_chi = cfg['N_chi']
N_rot = cfg['N_rot']
cor_offset = cfg["cor_offset"]
N_seg = N_theta*N_chi
filename = cfg['integrated_file']
wavelength_kev = cfg['wavelength']
min_two_theta = cfg['min_two_theta']
max_two_theta = cfg['max_two_theta']
reflection_cutoff = cfg['reflection_cutoff']

In [3]:
print(max_two_theta)

0.4


In [4]:
cif_folder = Path(cif_path)
filenames = [p.name for p in cif_folder.glob("*.cif")]
N_mat = len(filenames)
materials = []

for i_mat in range(N_mat):
    path = cif_folder / filenames[i_mat]
    material = Material.from_cif(
        cif_path=path,
        wavelength_kev=wavelength_kev,
        min_two_theta=min_two_theta,
        max_two_theta=max_two_theta,
        intensity_cutoff_fraction=reflection_cutoff,
    )
    materials.append(material)


grid_resolution_parameter = 6      # example
kernel_sigma = 0.25                # example
sigma_levels = [kernel_sigma]      # start with single level

grids = []

for mat in materials:
    tree = OrientationTree.from_hopf_fzone(
        mat.point_group_matrices,
        grid_resolution_parameter=grid_resolution_parameter,
        sigma_levels=sigma_levels,
    )
    grids.append(tree)

In [5]:
config_path = "configs/aluminum_config_small.yaml"
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

N_theta = 100
two_thetas = np.linspace(0.01,0.6,N_theta)

op = PFO_OPENCL_BATCHED(
        cfg = cfg,
        materials=materials,
        grids=grids,
        two_thetas = two_thetas,
        verbose=False)
op.set_pf_batch_max_gb(3.0)

# mem = MemoryCounter()
# memory_model = OperatorMemoryModel(op, mem)
# memory_model.mem.report(show_peak=True)
# memory_model.mem.report_allocations()
# memory_model.model_direct()
# memory_model.mem.report(show_peak=True)
# memory_model.mem.report_allocations()
# memory_model.model_adjoint()
# memory_model.mem.report(show_peak=True)
# memory_model.mem.report_allocations()

In [6]:
mem = MemoryCounter()

op_model = OperatorMemoryModel(op, mem)

fista = FISTAOpenCL(
    operator=op,
    prox_kind="nonneg_tv",
    lam=0.0,
    tau=1e-3
)

fista_mem = FISTAMemoryModel(fista, mem, op_model)

x_shape  = (op.Nx, op.Nx, op.K_sum)
Ax_shape = (op.N_rot, op.Nx, op.N_chi * op.N_theta)

fista_mem.model_run(x_shape, Ax_shape, niter=1)

mem.report()
mem.report_peak_allocations(min_mb=50)


=== MemoryCounter Report ===
Current : 3.226 GB
Peak    : 3.548 GB
Live allocations : 21

=== Peak Allocation Breakdown ===
Name                                        Size (MB)
-------------------------------------------------------
fista.Ax                                     1524.353
fista.r                                      1524.353
_out_sub[0]                                   228.653
data_gpu_sub[m0]                              228.653
-------------------------------------------------------
TOTAL @ PEAK                                 3506.012

